# 第4章 探索性分析与可视化 —— 配套代码

对应正文 `book/part1/04-eda-visualization.md`。依赖内置数据，离线可跑。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()
rets = daily_returns(load_sample_prices())
rets.describe().round(4)

## 偏度与峰度（厚尾的数字证据）

In [ ]:
import pandas as pd
summary = pd.DataFrame({"偏度": rets.skew(), "超额峰度": rets.kurtosis()})
summary.round(3)   # 超额峰度>0 即比正态更厚尾

## QQ 图：诊断厚尾

In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.ravel(), rets.columns):
    stats.probplot(rets[col], dist="norm", plot=ax)
    ax.set_title(f"{col} 正态 QQ 图")
plt.tight_layout()
plt.show()

## 波动率聚集：收益 vs 收益绝对值的自相关

In [ ]:
def acf(x, nlags=20):
    x = np.asarray(x) - np.mean(x)
    denom = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k or None]) / denom for k in range(1, nlags + 1)])

lags = range(1, 21)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(np.array(lags) - 0.2, acf(rets["TECH"]), width=0.4, label="收益率")
ax.bar(np.array(lags) + 0.2, acf(rets["TECH"].abs()), width=0.4, label="收益率绝对值")
ax.axhline(0, color="k", lw=0.8)
ax.set_title("TECH 自相关：收益近似不相关，但绝对值有持续自相关（波动率聚集）")
ax.set_xlabel("滞后阶数"); ax.legend()
plt.tight_layout(); plt.show()

## 相关性热力图

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(rets.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("四只股票日收益相关性")
plt.tight_layout(); plt.show()

## 滚动相关系数（相关性时变）

In [ ]:
roll = rets["BANK"].rolling(60).corr(rets["TECH"])
fig, ax = plt.subplots(figsize=(9, 4))
roll.plot(ax=ax)
ax.set_title("BANK 与 TECH 的 60 日滚动相关系数")
ax.set_ylabel("相关系数"); ax.set_xlabel("日期")
plt.tight_layout(); plt.show()